# DS 4320 Project 1: Predicting the Men's March Madness Tournament

## Data Pipeline File

In [1]:
# Import packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import seaborn as sns
import re
import html
import time
import requests
from IPython.display import display
import duckdb
import logging

In [13]:
# initiate logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Select completed seasons only; omit 2019-20 because no tournament was held
today = pd.Timestamp.today().normalize()
latest_completed_end_year = today.year if today.month >= 5 else today.year - 1
season_end_years = [y for y in range(2015, latest_completed_end_year + 1) if y != 2020]
# Runtime controls
MAX_SEASONS = None  # e.g., 3 for a quick test; None = all

# Reuse one HTTP session for reliability/performance
HTTP = requests.Session()
HTTP.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; DS4320/1.0)",
    "Accept": "application/json,text/html,*/*",
})


def season_label(end_year):
    return f"{end_year - 1}-{str(end_year)[-2:]}"


def flatten_columns(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [" ".join([str(x) for x in c if str(x) != "nan"]).strip() for c in df.columns]
    return df


def find_col_by_terms(df, include_terms, exclude_terms=None):
    exclude_terms = exclude_terms or []
    for c in df.columns:
        lc = c.lower().strip()
        if all(term in lc for term in include_terms) and not any(term in lc for term in exclude_terms):
            logger.debug(f"Matched column '{c}' for include_terms={include_terms} exclude_terms={exclude_terms}")
            return c
    return None


TEAM_ALIASES = {
    "lsu": "louisiana state",
    "ole miss": "mississippi",
    "vcu": "virginia commonwealth",
    "n c central": "north carolina central",
    "northern ky": "northern kentucky",
    "byu": "brigham young",
    "coastal caro": "coastal carolina",
    "eastern wash": "eastern washington",
    "north carolina state": "n c state",
    "smu": "southern methodist",
    "uni": "northern iowa",
    "bakersfield": "cal state bakersfield",
    "fairleigh dickinson": "fdu",
    "fgcu": "florida gulf coast",
    "middle tenn": "middle tennessee",
    "southern u": "southern",
    "ualr": "little rock",
    "uconn": "connecticut",
    "east tenn state": "east tennessee state",
    "mt state mary s": "mount st mary s",
    "state mary s ca": "saint mary s ca",
    "col of charleston": "charleston",
    "loyola chicago": "loyola il",
    "penn": "pennsylvania",
    "sfa": "stephen f austin",
    "tcu": "texas christian",
    "umbc": "maryland baltimore county",
    "prairie view": "prairie view aandm",
    "saint mary s ca": "saint mary s",
    "aandm corpus christi": "texas aandm corpus christi",
    "fla atlantic": "florida atlantic",
    "southeast mo state": "southeast missouri state",
    "western ky": "western kentucky",
    "siue": "southern illinois edwardsville",
    "uncw": "unc wilmington",
}


def normalize_team_name(name):
    if pd.isna(name):
        return ""
    s = str(name).lower().strip()
    s = s.replace("&", "and")
    s = re.sub(r"\b(ncaa|nit|cbi|cit)\b", " ", s)
    s = re.sub(r"\buniv\.?\b", "university", s)
    s = re.sub(r"\bst\.?\b", "state", s)
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    # Remove institutional suffix words so 'Baylor' and 'Baylor University' match
    s = re.sub(r"\b(university|college)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return TEAM_ALIASES.get(s, s)


def round_to_stage(round_name):
    # NCAA round labels vary by season and may include HTML entities/registered symbols.
    r = html.unescape(str(round_name).lower())
    r = re.sub(r"[^a-z0-9 ]+", " ", r)
    r = re.sub(r"\s+", " ", r).strip()

    if not r:
        return ""
    if "first four" in r:
        return "First Four"
    if "round of 64" in r or "first round" in r:
        return "Round of 64"
    if "round of 32" in r or "second round" in r or "third round" in r:
        return "Round of 32"
    if "sweet 16" in r or "regional semifinal" in r:
        return "Sweet 16"
    if "elite eight" in r or "elite 8" in r or "regional final" in r:
        return "Elite 8"
    if "national semifinal" in r or "final four" in r:
        return "Final Four"
    if "championship" in r or "title" in r or "national final" in r:
        return "National Championship"
    return ""


def stage_rank(stage):
    order = {
        "First Four": 1,
        "Round of 64": 2,
        "Round of 32": 3,
        "Sweet 16": 4,
        "Elite 8": 5,
        "Final Four": 6,
        "National Championship": 7,
    }
    return order.get(stage, 0)


def fetch_net_table(end_year):
    url = f"https://www.warrennolan.com/basketball/{end_year}/net"
    net_df = pd.read_html(url)[0].copy()
    net_df.columns = [str(c).strip() for c in net_df.columns]
    if "Team" not in net_df.columns or "NET Rank" not in net_df.columns:
        raise ValueError(f"Unexpected NET table columns: {list(net_df.columns)}")
    out = pd.DataFrame({
        "Season": season_label(end_year),
        "Team": net_df["Team"].astype(str).str.strip(),
        "NETRank": pd.to_numeric(net_df["NET Rank"], errors="coerce")
    })
    out["TeamKey"] = out["Team"].map(normalize_team_name)
    return out


def fetch_ncaa_tournament_table(end_year):
    # Start at Mar 13 to avoid non-NCAA postseason games that appear in early March scoreboards.
    days = pd.date_range(f"{end_year}-03-13", f"{end_year}-04-20", freq="D")
    rows = []
    for d in days:
        url = f"https://data.ncaa.com/casablanca/scoreboard/basketball-men/d1/{d:%Y/%m/%d}/scoreboard.json"
        try:
            resp = HTTP.get(url, timeout=20)
            if resp.status_code != 200:
                continue
            data = resp.json()
            for entry in data.get("games", []):
                g = entry.get("game", {})
                stage = round_to_stage(g.get("bracketRound", ""))
                for side in ["home", "away"]:
                    t = g.get(side, {}) or {}
                    names = t.get("names", {}) or {}
                    team_name = names.get("short") or names.get("full")
                    if not team_name:
                        continue
                    seed_raw = t.get("seed")
                    try:
                        seed = int(seed_raw) if str(seed_raw).strip() != "" else pd.NA
                    except Exception:
                        seed = pd.NA
                    if pd.isna(seed):
                        continue
                    rows.append({
                        "Season": season_label(end_year),
                        "Team": str(team_name).strip(),
                        "TeamKey": normalize_team_name(team_name),
                        "TournamentSeed": seed,
                        "RoundPlayed": stage,
                        "RoundRank": stage_rank(stage),
                    })
        except Exception:
            logger.error(f"Error occurred while fetching NCAA tournament data for {end_year} on {d:%Y-%m-%d}")
            continue

    if not rows:
        return pd.DataFrame(columns=["Season", "TeamKey", "MadeNCAATournament", "TournamentSeed", "TournamentFinishRound"])

    gdf = pd.DataFrame(rows)
    summary = gdf.groupby(["Season", "TeamKey"], as_index=False).agg(
        TournamentSeed=("TournamentSeed", "min"),
        FurthestRank=("RoundRank", "max"),
    )
    rank_to_name = {
        1: "First Four",
        2: "Round of 64",
        3: "Round of 32",
        4: "Sweet 16",
        5: "Elite 8",
        6: "Final Four",
        7: "National Championship",
    }
    summary["TournamentFinishRound"] = summary["FurthestRank"].map(rank_to_name).fillna("Unknown")
    summary["MadeNCAATournament"] = True
    return summary[["Season", "TeamKey", "MadeNCAATournament", "TournamentSeed", "TournamentFinishRound"]]


if MAX_SEASONS is not None:
    target_end_years = season_end_years[:MAX_SEASONS]
else:
    target_end_years = season_end_years

team_season_stats_rows = []
failed_seasons = []
net_rows = []
tourney_rows = []

overall_start = time.time()
for i, end_year in enumerate(target_end_years, start=1):
    label = season_label(end_year)
    season_start = time.time()
    print(f"[{i}/{len(target_end_years)}] Starting {label}...")
    logger.info(f"Starting data fetch for {label} (end_year={end_year})")

    try:
        url = f"https://www.basketball-reference.com/cbb/seasons/men/{end_year}-school-stats.html"
        season_df = pd.read_html(url)[0]
        season_df = flatten_columns(season_df)

        team_col = find_col_by_terms(season_df, ["school"])
        wins_col = find_col_by_terms(season_df, ["overall", "w"], exclude_terms=["w-l%", "srs", "sos"])
        losses_col = find_col_by_terms(season_df, ["overall", "l"], exclude_terms=["w-l%", "srs", "sos"])
        winpct_col = find_col_by_terms(season_df, ["w-l%"])
        games_col = find_col_by_terms(season_df, ["overall", "g"])
        srs_col = find_col_by_terms(season_df, ["overall", "srs"])
        sos_col = find_col_by_terms(season_df, ["overall", "sos"])

        conf_w_col = find_col_by_terms(season_df, ["conf", "w"], exclude_terms=["w-l%"])
        conf_l_col = find_col_by_terms(season_df, ["conf", "l"], exclude_terms=["w-l%"])
        home_w_col = find_col_by_terms(season_df, ["home", "w"], exclude_terms=["w-l%"])
        home_l_col = find_col_by_terms(season_df, ["home", "l"], exclude_terms=["w-l%"])
        away_w_col = find_col_by_terms(season_df, ["away", "w"], exclude_terms=["w-l%"])
        away_l_col = find_col_by_terms(season_df, ["away", "l"], exclude_terms=["w-l%"])

        pts_col = find_col_by_terms(season_df, ["points", "tm"])
        opp_pts_col = find_col_by_terms(season_df, ["points", "opp"])

        mp_col = find_col_by_terms(season_df, ["totals", "mp"])
        fg_col = find_col_by_terms(season_df, ["totals", "fg"], exclude_terms=["fg%", "fga"])
        fga_col = find_col_by_terms(season_df, ["totals", "fga"])
        fg_pct_col = find_col_by_terms(season_df, ["totals", "fg%"])
        tp_col = find_col_by_terms(season_df, ["totals", "3p"], exclude_terms=["3p%", "3pa"])
        tpa_col = find_col_by_terms(season_df, ["totals", "3pa"])
        tp_pct_col = find_col_by_terms(season_df, ["totals", "3p%"])
        ft_col = find_col_by_terms(season_df, ["totals", "ft"], exclude_terms=["ft%", "fta"])
        fta_col = find_col_by_terms(season_df, ["totals", "fta"])
        ft_pct_col = find_col_by_terms(season_df, ["totals", "ft%"])
        orb_col = find_col_by_terms(season_df, ["totals", "orb"])
        trb_col = find_col_by_terms(season_df, ["totals", "trb"])
        ast_col = find_col_by_terms(season_df, ["totals", "ast"])
        stl_col = find_col_by_terms(season_df, ["totals", "stl"])
        blk_col = find_col_by_terms(season_df, ["totals", "blk"])
        tov_col = find_col_by_terms(season_df, ["totals", "tov"])
        pf_col = find_col_by_terms(season_df, ["totals", "pf"])

        if team_col is None:
            raise ValueError(f"Could not find team column. Columns: {list(season_df.columns)}")

        season_df = season_df[season_df[team_col].astype(str).str.lower() != "school"]

        def num_or_na(col_name):
            if col_name is None:
                return pd.Series(pd.NA, index=season_df.index)
            return pd.to_numeric(season_df[col_name], errors="coerce")

        work = pd.DataFrame()
        work["Team"] = season_df[team_col].astype(str).str.strip()
        work = work[work["Team"].str.lower().ne("nan") & work["Team"].str.len().gt(0)].copy()
        work["TeamKey"] = work["Team"].map(normalize_team_name)
        work["Wins"] = num_or_na(wins_col)
        work["Losses"] = num_or_na(losses_col)
        work["Games"] = num_or_na(games_col)
        work["SRS"] = num_or_na(srs_col)
        work["SOS"] = num_or_na(sos_col)
        work["ConfWins"] = num_or_na(conf_w_col)
        work["ConfLosses"] = num_or_na(conf_l_col)
        work["HomeWins"] = num_or_na(home_w_col)
        work["HomeLosses"] = num_or_na(home_l_col)
        work["AwayWins"] = num_or_na(away_w_col)
        work["AwayLosses"] = num_or_na(away_l_col)

        if games_col is None:
            work["Games"] = work["Wins"].fillna(0) + work["Losses"].fillna(0)

        work["PointsForTotal"] = num_or_na(pts_col)
        work["PointsAgainstTotal"] = num_or_na(opp_pts_col)
        denom = work["Games"].replace(0, pd.NA)
        work["AvgPointsFor"] = work["PointsForTotal"] / denom
        work["AvgPointsAgainst"] = work["PointsAgainstTotal"] / denom

        if winpct_col is not None:
            work["WinPct"] = pd.to_numeric(season_df[winpct_col], errors="coerce")
        else:
            w_denom = work["Wins"].fillna(0) + work["Losses"].fillna(0)
            work["WinPct"] = work["Wins"] / w_denom.replace(0, pd.NA)

        work["TotalsMP"] = num_or_na(mp_col)
        work["TotalsFG"] = num_or_na(fg_col)
        work["TotalsFGA"] = num_or_na(fga_col)
        work["TotalsFGPct"] = num_or_na(fg_pct_col)
        work["Totals3P"] = num_or_na(tp_col)
        work["Totals3PA"] = num_or_na(tpa_col)
        work["Totals3PPct"] = num_or_na(tp_pct_col)
        work["TotalsFT"] = num_or_na(ft_col)
        work["TotalsFTA"] = num_or_na(fta_col)
        work["TotalsFTPct"] = num_or_na(ft_pct_col)
        work["TotalsORB"] = num_or_na(orb_col)
        work["TotalsTRB"] = num_or_na(trb_col)
        work["TotalsAST"] = num_or_na(ast_col)
        work["TotalsSTL"] = num_or_na(stl_col)
        work["TotalsBLK"] = num_or_na(blk_col)
        work["TotalsTOV"] = num_or_na(tov_col)
        work["TotalsPF"] = num_or_na(pf_col)

        # Fill average and helper columns from available totals so these are not left as N/A.
        work["HomeGames"] = (work["HomeWins"].fillna(0) + work["HomeLosses"].fillna(0)).replace(0, pd.NA)
        work["AvgPTS_Box"] = work["PointsForTotal"] / denom
        work["AvgREB_Box"] = work["TotalsTRB"] / denom
        work["AvgAST_Box"] = work["TotalsAST"] / denom
        work["Season"] = label

        try:
            net_df = fetch_net_table(end_year)
        except Exception as e:
            logger.warning(f"NET fetch unavailable for {label}: {e}")
            net_df = pd.DataFrame(columns=["Season", "Team", "TeamKey", "NETRank"])

        try:
            trn_df = fetch_ncaa_tournament_table(end_year)
        except Exception as e:
            logger.warning(f"Tournament fetch unavailable for {label}: {e}")
            trn_df = pd.DataFrame(columns=["Season", "TeamKey", "MadeNCAATournament", "TournamentSeed", "TournamentFinishRound"])

        net_rows.append(net_df)
        tourney_rows.append(trn_df)
        print(f"    Tournament team keys for {label}: {len(trn_df)}")
        logger.info(f"Fetched data for {label}: {len(work)} teams, NET={len(net_df)}, Tournament={len(trn_df)}")

        team_season_stats_rows.append(work)
        elapsed = time.time() - season_start
        print(f"[{i}/{len(target_end_years)}] Finished {label} in {elapsed:.1f}s ({len(work)} teams)")
        logger.info(f"Completed data processing for {label} in {elapsed:.1f}s")
        time.sleep(0.15)

    except Exception as e:
        failed_seasons.append((label, str(e)))
        elapsed = time.time() - season_start
        print(f"[{i}/{len(target_end_years)}] {label} failed in {elapsed:.1f}s: {e}")
        logger.info(f"Data fetch failed for {label} after {elapsed:.1f}s: {e}")

if team_season_stats_rows:
    team_season_stats = pd.concat(team_season_stats_rows, ignore_index=True)
    logger.info(f"Combined team season stats: {len(team_season_stats)} rows across {team_season_stats['Season'].nunique()} seasons")
else:
    team_season_stats = pd.DataFrame()

if net_rows:
    net_rankings = pd.concat(net_rows, ignore_index=True).drop_duplicates(["Season", "TeamKey"], keep="first")
    logger.info(f"Combined NET rankings: {len(net_rankings)} rows across {net_rankings['Season'].nunique()} seasons")
else:
    net_rankings = pd.DataFrame(columns=["Season", "Team", "TeamKey", "NETRank"])

if tourney_rows:
    ncaa_tournament_results = pd.concat(tourney_rows, ignore_index=True).drop_duplicates(["Season", "TeamKey"], keep="first")
    logger.info(f"Combined NCAA tournament results: {len(ncaa_tournament_results)} rows across {ncaa_tournament_results['Season'].nunique()} seasons")
else:
    ncaa_tournament_results = pd.DataFrame(columns=["Season", "TeamKey", "MadeNCAATournament", "TournamentSeed", "TournamentFinishRound"])

if not team_season_stats.empty:
    merged = team_season_stats.merge(
        net_rankings[["Season", "TeamKey", "NETRank"]],
        on=["Season", "TeamKey"],
        how="left"
    )
    merged = merged.merge(
        ncaa_tournament_results[["Season", "TeamKey", "MadeNCAATournament", "TournamentSeed", "TournamentFinishRound"]],
        on=["Season", "TeamKey"],
        how="left"
    )
    merged["MadeNCAATournament"] = merged["MadeNCAATournament"].fillna(False)
    merged = merged.sort_values(["Season", "Team"]).reset_index(drop=True)
    logger.info(f"Merged final dataset: {len(merged)} rows with {merged['Season'].nunique()} seasons and {merged['Team'].nunique()} teams")
else:
    merged = pd.DataFrame()

cbbpy_team_season_stats = merged

total_elapsed = time.time() - overall_start
print(f"\nRuntime this execution: {total_elapsed/60:.2f} minutes")
print(f"Seasons requested: {len(target_end_years)}")
print(f"Seasons loaded: {cbbpy_team_season_stats['Season'].nunique() if not cbbpy_team_season_stats.empty else 0}")
print(f"Team-season rows: {len(cbbpy_team_season_stats):,}")

if not cbbpy_team_season_stats.empty:
    display(cbbpy_team_season_stats.head(20))
    print("\nRelationship tables available:")
    print(f"- net_rankings rows: {len(net_rankings):,}")
    print(f"- ncaa_tournament_results rows: {len(ncaa_tournament_results):,}")
    print(f"- merged rows with MadeNCAATournament=True: {int(cbbpy_team_season_stats['MadeNCAATournament'].sum()):,}")
    print("\nTournament coverage by season:")
    print(
        cbbpy_team_season_stats.groupby("Season", as_index=False)["MadeNCAATournament"]
        .sum()
        .rename(columns={"MadeNCAATournament": "TournamentTeams"})
        .to_string(index=False)
    )

if failed_seasons:
    print("\nSeasons with issues:")
    for s, msg in failed_seasons:
        print(f"- {s}: {msg}")

2026-03-31 23:05:42,604 - INFO - Starting data fetch for 2014-15 (end_year=2015)


[1/10] Starting 2014-15...


2026-03-31 23:05:43,331 - WARNING - NET fetch unavailable for 2014-15: HTTP Error 404: Not Found
2026-03-31 23:05:50,441 - INFO - Fetched data for 2014-15: 351 teams, NET=0, Tournament=68
2026-03-31 23:05:50,443 - INFO - Completed data processing for 2014-15 in 7.8s
2026-03-31 23:05:50,595 - INFO - Starting data fetch for 2015-16 (end_year=2016)


    Tournament team keys for 2014-15: 68
[1/10] Finished 2014-15 in 7.8s (351 teams)
[2/10] Starting 2015-16...


2026-03-31 23:05:51,370 - WARNING - NET fetch unavailable for 2015-16: HTTP Error 404: Not Found
2026-03-31 23:05:57,166 - ERROR - Error occurred while fetching NCAA tournament data for 2016 on 2016-04-12
2026-03-31 23:05:59,983 - INFO - Fetched data for 2015-16: 351 teams, NET=0, Tournament=68
2026-03-31 23:05:59,985 - INFO - Completed data processing for 2015-16 in 9.4s
2026-03-31 23:06:00,138 - INFO - Starting data fetch for 2016-17 (end_year=2017)


...

    Tournament team keys for 2024-25: 68
[10/10] Finished 2024-25 in 8.0s (364 teams)


/tmp/ipykernel_88664/11295225.py:391: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged["MadeNCAATournament"] = merged["MadeNCAATournament"].fillna(False)
2026-03-31 23:07:15,955 - INFO - Merged final dataset: 3551 rows with 10 seasons and 580 teams



Runtime this execution: 1.56 minutes
Seasons requested: 10
Seasons loaded: 10
Team-season rows: 3,551


,Team,TeamKey,Wins,Losses,Games,SRS,SOS,ConfWins,ConfLosses,HomeWins,...,TotalsPF,HomeGames,AvgPTS_Box,AvgREB_Box,AvgAST_Box,Season,NETRank,MadeNCAATournament,TournamentSeed,TournamentFinishRound
0,Abilene Christian,abilene christian,10.0,31.0,31.0,-17.20,-6.34,4.0,14.0,7.0,...,661.0,13.0,63.741935,28.935484,12.709677,2014-15,NaN,False,NaN,NaN
1,Air Force,air force,14.0,31.0,31.0,-1.85,-0.71,6.0,12.0,11.0,...,552.0,16.0,65.709677,30.000000,15.580645,2014-15,NaN,False,NaN,NaN
2,Akron,akron,21.0,35.0,35.0,3.65,-0.47,9.0,9.0,15.0,...,654.0,18.0,68.057143,35.685714,12.428571,2014-15,NaN,False,NaN,NaN
3,Alabama,alabama,19.0,34.0,34.0,10.52,8.09,8.0,10.0,14.0,...,654.0,19.0,67.058824,33.000000,10.294118,2014-15,NaN,False,NaN,NaN
4,Alabama A&M,alabama aandm,9.0,29.0,29.0,-17.15,-9.62,8.0,10.0,4.0,...,573.0,11.0,61.965517,34.413793,12.206897,2014-15,NaN,False,NaN,NaN
5,Alabama State,alabama state,19.0,29.0,29.0,-8.97,-12.71,14.0,4.0,9.0,...,520.0,12.0,72.068966,39.689655,13.551724,2014-15,NaN,False,NaN,NaN
6,Albany (NY) NCAA,albany ny,24.0,33.0,33.0,-0.34,-5.18,15.0,1.0,12.0,...,544.0,15.0,65.333333,33.666667,10.575758,2014-15,NaN,True,14.0,Round of 32
7,Alcorn State,alcorn state,6.0,32.0,32.0,-20.75,-9.95,4.0,14.0,4.0,...,571.0,11.0,64.562500,32.437500,8.906250,2014-15,NaN,False,NaN,NaN
8,American,american,17.0,33.0,33.0,-3.56,-3.71,8.0,10.0,8.0,...,484.0,11.0,58.727273,25.060606,12.363636,2014-15,NaN,False,NaN,NaN
9,Appalachian State,appalachian state,12.0,29.0,29.0,-9.81,-3.18,9.0,11.0,7.0,...,551.0,12.0,63.482759,34.827586,10.965517,2014-15,NaN,False,NaN,NaN



Relationship tables available:
- net_rankings rows: 1,799
- ncaa_tournament_results rows: 680
- merged rows with MadeNCAATournament=True: 669

Tournament coverage by season:
 Season  TournamentTeams
2014-15               67
2015-16               68
2016-17               66
2017-18               67
2018-19               68
2020-21               67
2021-22               66
2022-23               67
2023-24               67
2024-25               66


In [14]:
# quick diagnostics after data fetch
avg_cols = ["AvgPointsFor", "AvgPointsAgainst", "AvgPTS_Box", "AvgREB_Box", "AvgAST_Box", "HomeGames"]
print("Failed seasons:", failed_seasons)
print("\nNull counts (overall):")
print(cbbpy_team_season_stats[avg_cols].isna().sum().to_string())
print("\nNull counts by season:")
print(cbbpy_team_season_stats.groupby("Season")[avg_cols].apply(lambda d: d.isna().sum()).to_string())

print("\nSample rows with missing AvgPointsFor:")
print(
    cbbpy_team_season_stats[cbbpy_team_season_stats["AvgPointsFor"].isna()][["Season", "Team", "Games", "PointsForTotal", "TotalsTRB", "TotalsAST"]]
    .head(40)
    .to_string(index=False)
)

Failed seasons: []

Null counts (overall):
AvgPointsFor        0
AvgPointsAgainst    0
AvgPTS_Box          0
AvgREB_Box          0
AvgAST_Box          0
HomeGames           2

Null counts by season:
         AvgPointsFor  AvgPointsAgainst  AvgPTS_Box  AvgREB_Box  AvgAST_Box  HomeGames
Season                                                                                
2014-15             0                 0           0           0           0          0
2015-16             0                 0           0           0           0          0
2016-17             0                 0           0           0           0          0
2017-18             0                 0           0           0           0          0
2018-19             0                 0           0           0           0          0
2020-21             0                 0           0           0           0          2
2021-22             0                 0           0           0           0          0
2022-23           

#### Develop the Database using DuckDB

ERD: 

In [15]:
# Ensure each team-season row has a stable primary key
if cbbpy_team_season_stats.empty:
    raise ValueError("cbbpy_team_season_stats is empty. Run the data creation cell first.")

cbbpy_team_season_stats = cbbpy_team_season_stats.copy()
cbbpy_team_season_stats["SquadID"] = (
    cbbpy_team_season_stats["Season"].astype(str) + "_" + cbbpy_team_season_stats["TeamKey"].astype(str)
)

# Load data into DuckDB tables
con = duckdb.connect(database="data/ncaam.db")
seasons = sorted(cbbpy_team_season_stats["Season"].dropna().unique().tolist())

# Core fact table
con.execute("""
    CREATE OR REPLACE TABLE team_season_stats AS
    SELECT * FROM cbbpy_team_season_stats
""")

# Dimension / relationship tables
con.execute("""
    CREATE OR REPLACE TABLE net_rankings AS
    SELECT DISTINCT Season, TeamKey, NETRank
    FROM net_rankings
""")
con.execute("""
    CREATE OR REPLACE TABLE ncaa_tournament_results AS
    SELECT DISTINCT Season, TeamKey, MadeNCAATournament, TournamentSeed, TournamentFinishRound
    FROM ncaa_tournament_results
""")

# Join table for season/team lookup
con.execute("""
    CREATE OR REPLACE TABLE join_table AS
    SELECT DISTINCT SquadID, TeamKey, Team, Season
    FROM cbbpy_team_season_stats
    ORDER BY Season, Team
""")

# Optional per-season tables for convenience
for season in seasons:
    season_df = cbbpy_team_season_stats[cbbpy_team_season_stats["Season"] == season]
    table_name = f"season_{season.replace('-', '_')}"
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM season_df
    """)
    logger.info(f"Loaded table {table_name} with {len(season_df)} rows")

logger.info("Data loading complete. Available tables:")
tables = con.execute("SHOW TABLES").fetchall()
for t in tables:
    logger.info(f"- {t[0]}")

2026-03-31 23:08:26,076 - INFO - Loaded table season_2014_15 with 351 rows
2026-03-31 23:08:26,102 - INFO - Loaded table season_2015_16 with 351 rows
2026-03-31 23:08:26,126 - INFO - Loaded table season_2016_17 with 351 rows
2026-03-31 23:08:26,155 - INFO - Loaded table season_2017_18 with 351 rows
2026-03-31 23:08:26,187 - INFO - Loaded table season_2018_19 with 353 rows
2026-03-31 23:08:26,243 - INFO - Loaded table season_2020_21 with 347 rows
2026-03-31 23:08:26,275 - INFO - Loaded table season_2021_22 with 358 rows
2026-03-31 23:08:26,295 - INFO - Loaded table season_2022_23 with 363 rows
2026-03-31 23:08:26,322 - INFO - Loaded table season_2023_24 with 362 rows
2026-03-31 23:08:26,341 - INFO - Loaded table season_2024_25 with 364 rows
2026-03-31 23:08:26,342 - INFO - Data loading complete. Available tables:
2026-03-31 23:08:26,370 - INFO - - join_table
2026-03-31 23:08:26,371 - INFO - - ncaa_tournament_results
2026-03-31 23:08:26,372 - INFO - - net_rankings
2026-03-31 23:08:26,373

#### Convert DB to Parquet files for easy storage

In [16]:
# convert db tables to parquet files
from pathlib import Path

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

for t in tables:
    table_name = t[0]
    df = con.execute(f"SELECT * FROM {table_name}").fetchdf()
    parquet_path = output_dir / f"{table_name}.parquet"
    df.to_parquet(parquet_path)
    logger.info(f"Exported {table_name} to {parquet_path}")

# export data to single csv for size check
all_data = con.execute("SELECT * FROM team_season_stats").fetchdf()
output_path = output_dir / "ncaam_data.csv"
all_data.to_csv(output_path, index=False)
logger.info(f"Exported all data to {output_path} ({len(all_data)} rows)")

2026-03-31 23:10:59,391 - INFO - Exported join_table to data/join_table.parquet
2026-03-31 23:10:59,414 - INFO - Exported ncaa_tournament_results to data/ncaa_tournament_results.parquet
2026-03-31 23:10:59,429 - INFO - Exported net_rankings to data/net_rankings.parquet
2026-03-31 23:10:59,460 - INFO - Exported season_2014_15 to data/season_2014_15.parquet
2026-03-31 23:10:59,482 - INFO - Exported season_2015_16 to data/season_2015_16.parquet
2026-03-31 23:10:59,502 - INFO - Exported season_2016_17 to data/season_2016_17.parquet
2026-03-31 23:10:59,524 - INFO - Exported season_2017_18 to data/season_2017_18.parquet
2026-03-31 23:10:59,544 - INFO - Exported season_2018_19 to data/season_2018_19.parquet
2026-03-31 23:10:59,571 - INFO - Exported season_2020_21 to data/season_2020_21.parquet
2026-03-31 23:10:59,589 - INFO - Exported season_2021_22 to data/season_2021_22.parquet
2026-03-31 23:10:59,615 - INFO - Exported season_2022_23 to data/season_2022_23.parquet
2026-03-31 23:10:59,636 - 

In [18]:
# release DuckDB lock for downstream notebooks
try:
    con.close()
    print("Closed existing DuckDB connection.")
except Exception as e:
    print(f"No open DuckDB connection to close: {e}")

Closed existing DuckDB connection.
